# 02 --- Context Isolation

**CCA Pattern**: Subagents do NOT inherit coordinator context. Each starts blank -- it receives ONLY what was explicitly passed.

This is the concept that trips up more candidates than any other.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))
sys.path.insert(0, str(Path('.').resolve()))

In [ ]:
from research_agents.models.research import SubTask
from research_agents.agent.context_builder import build_subagent_context
from research_agents.agent.agent_loop import AgentResult

## How Context Isolation Works

The `build_subagent_context()` function in `agent/context_builder.py` is the **only** way to create subagent input. It builds a plain string from three sources:

1. **`task.instruction`** -- what to do (always present)
2. **`task.context`** -- explicitly selected facts from the coordinator
3. **Predecessor results** -- filtered to only `task.depends_on` task IDs

What the subagent does **NOT** receive:
- The coordinator's message history
- The coordinator's system prompt
- Other subagents' results (unless listed in `depends_on`)
- The original research query (unless the coordinator chose to include it)

This is **structural isolation** -- the function signature makes it impossible to accidentally leak coordinator state:

```python
def build_subagent_context(
    task: SubTask,
    predecessor_results: dict[str, AgentResult] | None = None,
) -> str:
```

The coordinator's `messages` list is not even a parameter. You cannot pass it.

## Worked Example: Why Subagents Return MLA When Told to Use APA

This is the canonical CCA exam scenario for context isolation. Read it carefully before the anti-pattern cell below -- the sequence of events is exactly what the exam tests.

**The setup:**

1. A user asks the coordinator: *"Research renewable energy adoption. Use APA citation format for all sources."*
2. The coordinator reasons: "I'll decompose this into web research, document analysis, and fact checking."
3. The coordinator creates three `SubTask` objects. Each `SubTask.instruction` says "Search for renewable energy data" (or similar). **The APA formatting requirement never makes it into any `SubTask.context`.**
4. The web researcher runs, does a fine job finding sources, and returns citations formatted in MLA because MLA is its statistical default.
5. The user sees the final report, notices the citations are wrong, and reports it as a bug.

**Why the exam calls this the most-missed pattern:** it looks like the subagent disobeyed the coordinator. It didn't. It never received the instruction. The APA requirement was in the coordinator's turn-1 message, which the subagent structurally cannot see. The instruction lived in a place the subagent has no access to.

**The fix is explicit forwarding.** When the coordinator decomposes, anything that applies to all subagents (citation format, tone, date range, output schema) must be copied into every `SubTask.context` where it applies. No inheritance. No magic. Just explicit passing.

## Anti-Pattern: Shared Context

The coordinator passes its full message history to the subagent.

In [ ]:
from research_agents.anti_patterns.shared_context import run_leaky_subagent

# Simulate coordinator messages
coordinator_messages = [
    {'role': 'user', 'content': 'Research renewable energy globally'},
    {'role': 'assistant', 'content': 'I will decompose this into subtasks...'},
    {'role': 'user', 'content': 'Use APA citation format for all sources'},
]

# The leaky function converts messages to str() -- wasteful and polluting
leaked_context = str(coordinator_messages)
print(f'Leaked context length: {len(leaked_context)} chars')
print(f'Contains APA instruction: {"APA" in leaked_context}')
print(f'Contains coordinator reasoning: {"decompose" in leaked_context}')

### Why This Fails

The `run_leaky_subagent()` function in `anti_patterns/shared_context.py` does this:

```python
leaked_context = str(coordinator_messages)  # WRONG
```

Problems:
- **Token waste**: the subagent processes the entire coordinator conversation
- **Context pollution**: a web researcher sees document analysis instructions
- **Attention dilution**: useful context is buried in irrelevant messages
- **No isolation**: the subagent sees other subagents' results

## Correct Pattern: Explicit Context Passing

The context builder passes ONLY what the coordinator deliberately selects.

In [ ]:
# Correct: SubTask contains only explicit context
task = SubTask(
    task_id='t1',
    agent_type='web_researcher',
    instruction='Search for renewable energy adoption statistics',
    context='Focus on 2024 data from government and peer-reviewed sources. Use APA citation format.',
)

explicit_context = build_subagent_context(task)
print(f'Explicit context length: {len(explicit_context)} chars')
print(f'Contains instruction: {"renewable energy" in explicit_context}')
print(f'Contains APA format: {"APA" in explicit_context}')
print(f'Contains coordinator reasoning: {"decompose" in explicit_context}')
print()
print(explicit_context)

### Predecessor Results: The `depends_on` Filter

When a task depends on earlier tasks, the context builder includes only those specific results -- not all prior results:


In [ ]:
# Simulate predecessor results from Wave 0
prior_results = {
    't1': AgentResult(content='Found 4 sources on renewable energy.'),
    't2': AgentResult(content='Database shows 1580 GW solar capacity in 2024.'),
    't3': AgentResult(content='Document analysis of IEA report complete.'),
}

# Fact checker depends on t1 and t2 only -- NOT t3
fact_check_task = SubTask(
    task_id='t4',
    agent_type='fact_checker',
    instruction='Verify renewable energy claims',
    context='Cross-reference web and database findings',
    depends_on=['t1', 't2'],  # Only these results are passed
)

context_with_deps = build_subagent_context(fact_check_task, prior_results)
print(context_with_deps)
print()
print(f'Contains t1 result: {"Found 4 sources" in context_with_deps}')
print(f'Contains t2 result: {"1580 GW" in context_with_deps}')
print(f'Contains t3 result: {"IEA report" in context_with_deps}')  # Should be False

In [ ]:
from helpers import compare_results

compare_results(
    {'context_length': len(leaked_context), 'contains_coordinator_reasoning': True, 'contains_other_agent_results': True, 'type': 'str(messages)'},
    {'context_length': len(explicit_context), 'contains_coordinator_reasoning': False, 'contains_other_agent_results': False, 'type': 'explicit_string'},
)

## CCA Exam Tip

> Any question where a subagent produces results that 'should have followed the coordinator\'s instructions' is testing context isolation.
> - The answer is always that the instructions were in the coordinator's context but never explicitly forwarded
> - Subagents do not inherit. Subagents receive only what you explicitly send.
> - The `build_subagent_context()` pattern enforces this structurally -- the coordinator's messages are never a function parameter